In [28]:
import os, sys, math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from pathlib import Path

project_root = Path.cwd()
while not (project_root / 'requirements.txt').exists():
    if project_root == project_root.parent:
        raise RuntimeError("Could not find project root (looked for requirements.txt)")
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.models.main.classical import ShallowCNN, ClassicalModel
from src.models.main.models import QuantumModel, ExtentedQuantumModel
from src.models.fusion import OnlineFusionModel
from src.utils.dataloader import BreastMNISTLoader, INbreastLoader, BUSUCLMLoader
from src.utils.training import train_epoch, validate_model, test_model, training_loop

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Project root: {project_root}")

Device: cuda
Project root: /home/eflammere/HybridEmbeddingsForBreastCancerDetection


In [ ]:
DATASET = 'BUSUCLM'
CLASSICAL_MODEL_TYPE = 'shallow'
QUANTUM_MODEL_TYPE = 'extended_quantum'

CLASSICAL_PRETRAINED_PATH = str(project_root / 'experiments' / 'models' / 'classical' / DATASET / CLASSICAL_MODEL_TYPE / f'best_{CLASSICAL_MODEL_TYPE}.pth')
QUANTUM_PRETRAINED_PATH = str(project_root / 'experiments' / 'models' / 'quantum' / QUANTUM_MODEL_TYPE / DATASET / 'checkpoints' / 'model_epoch18.pth')

FREEZE_CLASSICAL = False
FREEZE_QUANTUM = False

BATCH_SIZE = 32
IMAGE_SIZE = 28
NUM_EPOCHS = 25
FEATURE_DIM = 128
NUM_CLASSES = 2

LR_CLASSICAL = 1e-4
LR_QUANTUM = 1e-2
LR_FUSION = 1e-1
LR_GAMMA = 1e-0
INIT_GAMMA = 1

N_QUBITS = 4
N_LAYERS = 1
PATCH_SIZE = 2

OUTPUT_DIR = str(project_root / 'experiments' / 'models' / 'fusion' / 'online' / DATASET / f'{CLASSICAL_MODEL_TYPE}_{QUANTUM_MODEL_TYPE}' / 'temperature')
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [30]:
if DATASET == 'BreastMNIST':
    loader = BreastMNISTLoader(batch_size=BATCH_SIZE, image_size=IMAGE_SIZE)
elif DATASET == 'INbreast':
    loader = INbreastLoader(batch_size=BATCH_SIZE, image_size=IMAGE_SIZE,
                            root_dir=str(project_root / 'src' / 'data' / 'INbreast' / 'png'))
elif DATASET == 'BUSUCLM':
    loader = BUSUCLMLoader(batch_size=BATCH_SIZE, image_size=IMAGE_SIZE,
                           root_dir=str(project_root / 'src' / 'data' / 'BUS-UCLM' / 'png'))
else:
    raise ValueError(f"Dataset não suportado: {DATASET}")

train_loader, test_loader, val_loader = loader.get_dataloaders()
print(f"{DATASET} — Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

Loaded 165 images from /home/eflammere/HybridEmbeddingsForBreastCancerDetection/src/data/BUS-UCLM/png/train/0
Loaded 45 images from /home/eflammere/HybridEmbeddingsForBreastCancerDetection/src/data/BUS-UCLM/png/train/1
Loaded 2 images from /home/eflammere/HybridEmbeddingsForBreastCancerDetection/src/data/BUS-UCLM/png/val/0
Loaded 26 images from /home/eflammere/HybridEmbeddingsForBreastCancerDetection/src/data/BUS-UCLM/png/val/1
Loaded 7 images from /home/eflammere/HybridEmbeddingsForBreastCancerDetection/src/data/BUS-UCLM/png/test/0
Loaded 19 images from /home/eflammere/HybridEmbeddingsForBreastCancerDetection/src/data/BUS-UCLM/png/test/1
BUSUCLM — Train: 7 | Val: 1 | Test: 1


In [31]:
class TemperatureFusionModel(OnlineFusionModel):
    def __init__(self, classical_model, quantum_model, feature_dim=128,
                 num_classes=2, freeze_classical=False, freeze_quantum=False,
                 init_gamma=1.0):
        super().__init__(classical_model=classical_model, quantum_model=quantum_model,
                         feature_dim=feature_dim, num_classes=num_classes,
                         freeze_classical=freeze_classical, freeze_quantum=freeze_quantum)
        init_gamma = float(max(init_gamma, 1e-6))
        self.log_gamma = nn.Parameter(torch.tensor(math.log(init_gamma), dtype=torch.float32))

    @property
    def gamma(self):
        return torch.exp(self.log_gamma)

    def extract_features(self, x):
        if hasattr(self.classical_model, 'extract_features'):
            classical_feats = self.classical_model.extract_features(x)
        else:
            classical_feats = self.classical_model(x)
        if hasattr(self.quantum_model, 'extract_features'):
            quantum_feats = self.quantum_model.extract_features(x)
        else:
            quantum_feats = self.quantum_model(x)

        if classical_feats.ndim > 2:
            classical_feats = classical_feats.flatten(start_dim=1)
        if quantum_feats.ndim > 2:
            quantum_feats = quantum_feats.flatten(start_dim=1)
        if self.classical_projection is None or self.quantum_projection is None:
            self._init_projections(classical_feats.shape[1], quantum_feats.shape[1], x.device)

        classical_proj = self.classical_projection(classical_feats)
        quantum_proj = self.quantum_projection(quantum_feats)

        classical_norm = F.normalize(classical_proj, p=2, dim=1)
        quantum_norm = F.normalize(quantum_proj, p=2, dim=1)
        quantum_scaled = self.gamma * quantum_norm

        return torch.cat([classical_norm, quantum_scaled], dim=1)

    def get_parameter_groups(self, lr_classical=1e-4, lr_quantum=1e-3,
                             lr_fusion=1e-3, lr_gamma=1e-4):
        groups = super().get_parameter_groups(lr_classical, lr_quantum, lr_fusion)
        groups.append({'params': [self.log_gamma], 'lr': lr_gamma})
        return groups


# Modelo clássico
if CLASSICAL_MODEL_TYPE == 'shallow':
    classical_model = ShallowCNN(num_classes=NUM_CLASSES, input_channels=1)
elif CLASSICAL_MODEL_TYPE == 'resnet':
    classical_model = ClassicalModel(num_classes=NUM_CLASSES)

# Modelo quântico
if QUANTUM_MODEL_TYPE == 'quantum':
    rand_params = np.random.uniform(high=2*np.pi, size=(N_QUBITS, N_LAYERS))
    quantum_model = QuantumModel(
        n_qubits=N_QUBITS, n_layers=N_LAYERS, rand_params=rand_params,
        output_size=(14, 14), patch_size=PATCH_SIZE, num_classes=NUM_CLASSES)
elif QUANTUM_MODEL_TYPE == 'extended_quantum':
    quantum_model = ExtentedQuantumModel(
        n_qubits=N_QUBITS, n_layers=N_LAYERS, num_classes=NUM_CLASSES,
        patch_size=PATCH_SIZE, mode="2d", stride=PATCH_SIZE, use_parallel=True)

# Fusão com temperatura
fusion_model = TemperatureFusionModel(
    classical_model=classical_model, quantum_model=quantum_model,
    feature_dim=FEATURE_DIM, num_classes=NUM_CLASSES,
    freeze_classical=FREEZE_CLASSICAL, freeze_quantum=FREEZE_QUANTUM,
    init_gamma=INIT_GAMMA)

if CLASSICAL_PRETRAINED_PATH or QUANTUM_PRETRAINED_PATH:
    fusion_model.load_pretrained_weights(
        classical_path=CLASSICAL_PRETRAINED_PATH,
        quantum_path=QUANTUM_PRETRAINED_PATH, device=device)

fusion_model = fusion_model.to(device)

total_params = sum(p.numel() for p in fusion_model.parameters())
trainable_params = sum(p.numel() for p in fusion_model.parameters() if p.requires_grad)
print(f"Params: {trainable_params:,}/{total_params:,} | γ={fusion_model.gamma.item():.4f}")

Quantum layer has 4 trainable parameters
Requires grad: True
✓ Loaded classical model weights from /home/eflammere/HybridEmbeddingsForBreastCancerDetection/experiments/models/classical/BUSUCLM/shallow/best_shallow.pth
Params: 94,409/94,409 | γ=1.0000


/home/eflammere/HybridEmbeddingsForBreastCancerDetection/src/models/fusion.py:218: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(classical_path, map_

In [32]:
fusion_model.eval()
with torch.no_grad():
    _ = fusion_model(torch.randn(1, 1, IMAGE_SIZE, IMAGE_SIZE).to(device))
fusion_model.train()

param_groups = fusion_model.get_parameter_groups(
    lr_classical=LR_CLASSICAL, lr_quantum=LR_QUANTUM,
    lr_fusion=LR_FUSION, lr_gamma=LR_GAMMA)

optimizer = optim.Adam(param_groups)
criterion = nn.CrossEntropyLoss()

for i, (name, group) in enumerate(zip(['Classical', 'Quantum', 'Fusion', 'γ'], param_groups)):
    n = sum(p.numel() for p in group['params'])
    print(f"  {name}: lr={group['lr']}, params={n:,}")

  Classical: lr=0.0001, params=93,378
  Quantum: lr=0.01, params=4
  Fusion: lr=0.1, params=118,018
  γ: lr=0.1, params=1


In [35]:
history = training_loop(
    model=fusion_model, train_loader=train_loader, val_loader=val_loader,
    criterion=criterion, optimizer=optimizer, device=device,
    num_epochs=NUM_EPOCHS, save_dir=OUTPUT_DIR,
    model_name=f"fusion_{CLASSICAL_MODEL_TYPE}_{QUANTUM_MODEL_TYPE}_temperature",
    extract_features=False, save_best_only=True, save_all_epochs=False)

print(f"Gamma final: {fusion_model.gamma.item():.4f}")

Epoch 1/25



Epoch 1 Summary:
  Train - Loss: 1.3877, Acc: 0.5238
  Val   - Loss: 0.2650, Acc: 0.9286, Precision: 0.9286, Recall: 1.0000, F1: 0.9630, AUC: 0.7115
  ✓ New best model saved (val_acc: 0.9286) at /home/eflammere/HybridEmbeddingsForBreastCancerDetection/experiments/models/fusion/online/BUSUCLM/shallow_extended_quantum/temperature/best_fusion_shallow_extended_quantum_temperature.pth
Epoch 2/25



Epoch 2 Summary:
  Train - Loss: 0.8891, Acc: 0.5286
  Val   - Loss: 0.8820, Acc: 0.0714, Precision: 0.0000, Recall: 0.0000, F1: 0.0000, AUC: 0.5962
Epoch 3/25



Epoch 3 Summary:
  Train - Loss: 0.6267, Acc: 0.6381
  Val   - Loss: 0.6545, Acc: 0.6786, Precision: 0.9474, Recall: 0.6923, F1: 0.8000, AUC: 0.6346
Epoch 4/25


KeyboardInterrupt: 

In [36]:
best_path = os.path.join(OUTPUT_DIR, f"best_fusion_{CLASSICAL_MODEL_TYPE}_{QUANTUM_MODEL_TYPE}_temperature.pth")
if os.path.exists(best_path):
    fusion_model.load_state_dict(torch.load(best_path, map_location=device))

hyperparams = {
    'dataset': DATASET, 'classical_model': CLASSICAL_MODEL_TYPE,
    'quantum_model': QUANTUM_MODEL_TYPE, 'feature_dim': FEATURE_DIM,
    'lr_classical': LR_CLASSICAL, 'lr_quantum': LR_QUANTUM,
    'lr_fusion': LR_FUSION, 'lr_gamma': LR_GAMMA,
    'init_gamma': INIT_GAMMA, 'learned_gamma': fusion_model.gamma.item(),
    'n_qubits': N_QUBITS, 'n_layers': N_LAYERS, 'patch_size': PATCH_SIZE}

output_file = os.path.join(OUTPUT_DIR, 'test_results.txt')
test_loss, test_acc, test_prec, test_rec, test_f1, test_auc = test_model(
    model=fusion_model, dataloader=test_loader, criterion=criterion,
    device=device, hyperparams=hyperparams, output_file=output_file,
    extract_features=False, compute_detailed_metrics=True)

print(f"Loss={test_loss:.4f} Acc={test_acc:.4f} P={test_prec:.4f} R={test_rec:.4f} F1={test_f1:.4f} AUC={test_auc:.4f} γ={fusion_model.gamma.item():.4f}")

/tmp/ipykernel_409620/1384068345.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  fusion_model.load_state_dict(torch.load(best_path, map_location=device))
Testing: 100%|█

Test Loss: 0.8786
Accuracy: 0.7308, Precision: 0.7308, Recall: 1.0000, F1: 0.8444, AUC: 0.9098
Macro - Precision: 0.3654, Recall: 0.5000, F1: 0.4222
Micro - Precision: 0.7308, Recall: 0.7308, F1: 0.7308
Loss=0.8786 Acc=0.7308 P=0.7308 R=1.0000 F1=0.8444 AUC=0.9098 γ=1.0196


In [ ]:
import json

config = {
    'dataset': DATASET, 'classical_model': CLASSICAL_MODEL_TYPE,
    'quantum_model': QUANTUM_MODEL_TYPE, 'feature_dim': FEATURE_DIM,
    'num_classes': NUM_CLASSES, 'batch_size': BATCH_SIZE,
    'image_size': IMAGE_SIZE, 'num_epochs': NUM_EPOCHS,
    'lr_classical': LR_CLASSICAL, 'lr_quantum': LR_QUANTUM,
    'lr_fusion': LR_FUSION, 'lr_gamma': LR_GAMMA,
    'init_gamma': INIT_GAMMA, 'learned_gamma': fusion_model.gamma.item(),
    'n_qubits': N_QUBITS, 'n_layers': N_LAYERS, 'patch_size': PATCH_SIZE,
    'freeze_classical': FREEZE_CLASSICAL, 'freeze_quantum': FREEZE_QUANTUM,
    'total_params': total_params, 'trainable_params': trainable_params,
    'test_results': {
        'loss': float(test_loss), 'accuracy': float(test_acc),
        'precision': float(test_prec), 'recall': float(test_rec),
        'f1': float(test_f1), 'auc': float(test_auc)}}

with open(os.path.join(OUTPUT_DIR, 'config.json'), 'w') as f:
    json.dump(config, f, indent=4)
print("Done.")

Done.
